### 서울시 교통사고 시각화

In [1]:
import pandas as pd
import numpy as np
import koreanize_matplotlib
import matplotlib.pyplot as plt
import warnings
import folium
warnings.filterwarnings('ignore')

In [2]:
# 서울시 교통사고 데이터 불러오기
traffic_Seoul = pd.read_csv("../Data/newSeoul_2005_2019.csv")
traffic_Seoul.head()

,년도,월,자치구명,발생건수,사망자수,부상자수
0,2005,1,종로구,93,2,138
1,2005,2,종로구,84,3,125
2,2005,3,종로구,117,0,142
3,2005,4,종로구,138,2,212
4,2005,5,종로구,145,2,207


In [3]:
traffic_Seoul.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   년도      4500 non-null   int64 
 1   월       4500 non-null   int64 
 2   자치구명    4500 non-null   object
 3   발생건수    4500 non-null   int64 
 4   사망자수    4500 non-null   int64 
 5   부상자수    4500 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 211.1+ KB


In [4]:
# 2019년 데이터만 추출하기
traffic_Seoul_2019 = \
traffic_Seoul[traffic_Seoul.년도 == 2019]

traffic_Seoul_2019.head()

,년도,월,자치구명,발생건수,사망자수,부상자수
4200,2019,1,종로구,87,1,125
4201,2019,2,종로구,66,1,84
4202,2019,3,종로구,87,2,122
4203,2019,4,종로구,85,0,131
4204,2019,5,종로구,112,1,158


In [5]:
# index 정리하기

traffic_Seoul_2019.reset_index(
  drop=True,
  inplace=True
)

traffic_Seoul_2019.head()

,년도,월,자치구명,발생건수,사망자수,부상자수
0,2019,1,종로구,87,1,125
1,2019,2,종로구,66,1,84
2,2019,3,종로구,87,2,122
3,2019,4,종로구,85,0,131
4,2019,5,종로구,112,1,158


In [6]:
# traffic_anal = traffic_Seoul_2019['자치구명','부상자수','사망자수']
# traffic_anal.head()


traffic_anal = \
  traffic_Seoul_2019.pivot_table(
    ['발생건수','부상자수','사망자수'],
    index=['자치구명'],
    aggfunc='sum' # 빈도수
  )

traffic_anal.head()

,발생건수,부상자수,사망자수
자치구명,,,
강남구,3722,5182,14
강동구,1414,1910,11
강북구,1277,1706,7
강서구,1829,2491,20
관악구,1363,1755,10


In [7]:
traffic_anal = \
pd.pivot_table(
  traffic_Seoul_2019[['발생건수','부상자수','사망자수']],
  index=traffic_Seoul_2019['자치구명'],
  aggfunc='sum'
)
traffic_anal.head()


,발생건수,부상자수,사망자수
자치구명,,,
강남구,3722,5182,14
강동구,1414,1910,11
강북구,1277,1706,7
강서구,1829,2491,20
관악구,1363,1755,10


In [8]:
# 위도, 경로를 포함한 데이터 불러오기

seoul_limit = pd.read_csv("../Data/seoul.csv", encoding='euc-kr') # euc-kr로 인코딩 되어있다는 것.
seoul_limit.head()

,area,lon,lat
0,강남구,127.0475,37.51731
1,강동구,127.1238,37.53013
2,강북구,127.0255,37.63975
3,관악구,126.9515,37.47834
4,구로구,126.8875,37.49547


In [9]:
# seoul_limit의 컬럼 이름 변경하기
seoul_limit.rename(columns={'area': '자치구명'}, inplace=True)

seoul_limit.head()

,자치구명,lon,lat
0,강남구,127.0475,37.51731
1,강동구,127.1238,37.53013
2,강북구,127.0255,37.63975
3,관악구,126.9515,37.47834
4,구로구,126.8875,37.49547


In [10]:
# Merge
data_result = pd.merge(
  traffic_anal,
  seoul_limit,
  on='자치구명'
)

data_result.head()

,자치구명,발생건수,부상자수,사망자수,lon,lat
0,강남구,3722,5182,14,127.0475,37.51731
1,강동구,1414,1910,11,127.1238,37.53013
2,강북구,1277,1706,7,127.0255,37.63975
3,강서구,1829,2491,20,126.8496,37.55094
4,관악구,1363,1755,10,126.9515,37.47834


In [11]:
data_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   자치구명    25 non-null     object 
 1   발생건수    25 non-null     int64  
 2   부상자수    25 non-null     int64  
 3   사망자수    25 non-null     int64  
 4   lon     25 non-null     float64
 5   lat     25 non-null     float64
dtypes: float64(2), int64(3), object(1)
memory usage: 1.3+ KB


#### folium을 사용하기 위해선 int -> float로 변환이 필요

In [12]:
data_result['발생건수'] = data_result['발생건수'].astype(float)
data_result['부상자수'] = data_result['부상자수'].astype(float)
data_result['사망자수'] = data_result['사망자수'].astype(float)

data_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   자치구명    25 non-null     object 
 1   발생건수    25 non-null     float64
 2   부상자수    25 non-null     float64
 3   사망자수    25 non-null     float64
 4   lon     25 non-null     float64
 5   lat     25 non-null     float64
dtypes: float64(5), object(1)
memory usage: 1.3+ KB


In [13]:
# 자치구의 경찰서의 위치를 지도의 Marker로 표시하기

seoul_map = folium.Map(
  location=[37.55, 126.95],
  zoom_start = 12
)
seoul_map

In [14]:
# 서울 지도에 학교 위치 표시하기
seoul_map = folium.Map(
  location=[37.55, 126.98],
  zoom_start=12
)

# 위치 정보를 Marker로 표시
for 자치구명, lat, lon in zip(data_result.자치구명, data_result.lat, data_result.lon):
  popup = folium.Popup(자치구명, max_width=200)
  folium.Marker(
    [lat,lon],
    popup=popup
  ).add_to(seoul_map)

  folium.CircleMarker(
    [lat,lon],
    radius=10,
    color='brown', # 원의 둘레 색상
    fill_color='coral',
    fill=True,
    fill_opacity=0.7
  ).add_to(seoul_map)

seoul_map

#### 자치구별 교통사고 발생건수를 표시하기

In [15]:
# 서울 지도에 학교 위치 표시하기
seoul_map = folium.Map(
  location=[37.55, 126.98],
  zoom_start=12
)

# 대학교 위치 정보를 Marker로 표시
for 자치구명, lat, lon, radius1, radius2 in zip(data_result.자치구명, data_result.lat, data_result.lon, data_result.발생건수, data_result.사망자수):
  popup = folium.Popup(자치구명, max_width=200)
  folium.Marker(
    [lat,lon],
    popup=popup
  ).add_to(seoul_map)

  folium.CircleMarker(
    [lat,lon],
    radius=radius1/90,
    color='brown', # 원의 둘레 색상
    fill_color='coral',
    fill=True,
    fill_opacity=0.7
  ).add_to(seoul_map)


  folium.CircleMarker(
    [lat,lon],
    radius=radius2,
    color='skyblue', # 원의 둘레 색상
    fill_color='skyblue',
    fill=True,
    fill_opacity=0.7
  ).add_to(seoul_map)

seoul_map

In [16]:
np.corrcoef(
  data_result.사망자수,
  data_result.부상자수
)

array([[1.        , 0.47964054],
       [0.47964054, 1.        ]])